# Sentiment Analysis Using Bag of Words and Random Forest

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

In this project, I build a complete sentiment analysis pipeline using classical NLP techniques and Machine Learning algorithms. The workflow covers the entire process from raw text preprocessing to feature extraction, model training, evaluation, and experiment documentation.

The primary objective of this project is not only to achieve high predictive performance but also to systematically investigate how different preprocessing techniques, feature engineering strategies, and machine learning models affect sentiment classification performance.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:

* Positive
* Negative

The reviews contain real-world natural language, making them suitable for evaluating text preprocessing techniques and machine learning models.

---

## Project Pipeline

The following stages are implemented throughout this project:

### 1. Text Preprocessing

* Contraction expansion
* Lowercasing
* Text cleaning using regular expressions
* Stopword removal with preserved negations
* Lemmatization
* Corpus construction

### 2. Feature Extraction

* Bag of Words (CountVectorizer)
* N-gram generation
* Vocabulary analysis
* Feature selection using `max_features`
* Rare-word filtering using `min_df`

### 3. Model Training

Different machine learning algorithms are evaluated and compared, including:

* Gaussian Naive Bayes
* Multinomial Naive Bayes
* Bernoulli Naive Bayes
* Logistic Regression
* Support Vector Machines (SVM)
* Decision Trees
* Random Forests

### 4. Model Evaluation

Performance is assessed using multiple metrics:

* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single model, this notebook follows an experimentation-driven methodology.

For each model, multiple configurations are tested, including different values for:

* `max_features`
* `min_df`
* N-gram ranges
* Preprocessing strategies

The goal is to identify the most effective configuration while understanding the trade-offs between model complexity, computational cost, and predictive performance.

---

## Key Learning Objectives

Through this project, I aim to:

* Develop a deeper understanding of NLP preprocessing techniques.
* Compare the behavior of different machine learning algorithms on text data.
* Analyze how feature engineering impacts classification performance.
* Build reproducible NLP pipelines suitable for real-world applications.
* Establish strong baselines before moving toward advanced embedding and transformer-based approaches.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)


# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [ ]:
dataset = pd.read_csv('MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 5.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.0 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never'}
all_stopwords = base_stopwords - negation_words

wnl = WordNetLemmatizer()

corpus = []

for i in range(0, len(dataset)):
    review = dataset['review'][i]
    # Fix contractions
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split
    words = review.split()
    # Chained Lemmatization (Handles both Verbs 'v' and Nouns 'n')
    review = [wnl.lemmatize(wnl.lemmatize(word, pos='v'), pos='n') for word in words if word not in all_stopwords]
    review = ' '.join(review) 
    corpus.append(review)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Encoding Categorical data Using Label Encoding**

In [10]:
from sklearn.preprocessing import LabelEncoder
y = dataset.iloc[:, 0].values
le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [12]:
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size = 0.20, random_state = 0)

# **Creating the Bag of Word model**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# Add ngram_range=(1, 2) so it automatically catches phrases like "not good" or "no clue"
cv = CountVectorizer(max_features=10000, ngram_range=(1,2))
X_train = cv.fit_transform(X_train)
X_test = cv.transform(X_test)

In [15]:
print(X_train.shape)        # (n_samples, n_features)
print(X_train.shape[0])     # number of training samples
print(X_train.shape[1])     # number of features (vocabulary size)

(19923, 10000)
19923
10000


In [16]:
print(X_test.shape)        # (n_samples, n_features)
print(X_test.shape[0])     # number of training samples
print(X_test.shape[1])     # number of features (vocabulary size)

(4981, 10000)
4981
10000


# Tuning Hyperparameters with HalvingRandomSearchCV

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv  # Required
from sklearn.model_selection import HalvingRandomSearchCV

# Initialize Random Forest
rf = RandomForestClassifier(random_state=0)

# Define parameter grid (without using n_estimators as resource)
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 5, 10, 20, 30, 50],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False]
}

# Halving Random Search
halving_search = HalvingRandomSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    factor=3,              # controls how aggressively candidates are reduced
    resource='n_samples',  # progressively increase number of training samples
    max_resources='auto',  # automatically use full dataset at final stage
    scoring='accuracy',
    cv=3,                  # fewer folds for speed
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# Fit on training data
halving_search.fit(X_train, y_train)

# Best parameters and score
print("Best Parameters:", halving_search.best_params_)
print("Best CV Score:", halving_search.best_score_)


# **Training the Random Forest model on the Training set**

In [17]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=300, max_depth = None, min_samples_leaf = 1, min_samples_split = 5, max_features = 'sqrt', criterion='log_loss', bootstrap = False, random_state=0)
classifier.fit(X_train, y_train)

RandomForestClassifier(bootstrap=False, criterion='log_loss',
                       min_samples_split=5, n_estimators=300, random_state=0)

# **Predicting the Test set results**

In [18]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [1 0]
 ...
 [0 0]
 [0 0]
 [1 0]]


# **Evaluating the Model Performance**

In [19]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_pred))


accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2137  383]
 [ 308 2153]]

Accuracy Score:
0.8612728367797631

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      2520
           1       0.85      0.87      0.86      2461

    accuracy                           0.86      4981
   macro avg       0.86      0.86      0.86      4981
weighted avg       0.86      0.86      0.86      4981


ROC-AUC Score:
0.861431747966693

Mean Accuracy:
0.8571500276062842

Standard Deviation:
0.002728038596938479


# Random Forest Performance Analysis for Text Classification

# 1. Objective

The objective of this experiment was to evaluate the effectiveness of the **Random Forest** algorithm for binary text classification using features generated by **CountVectorizer**.

As an ensemble of Decision Trees, Random Forest aggregates the predictions of many individually weak, decorrelated trees to reduce the overfitting and instability that limit a single Decision Tree. This study investigates how vocabulary size and hyperparameter optimization influence performance, and directly tests whether ensembling closes the gap left by the single-tree results from the prior Decision Tree experiment.

The experiment aims to answer the following research questions:

* Does ensembling multiple trees meaningfully outperform a single Decision Tree on sparse text data?
* How does vocabulary size affect Random Forest performance?
* What role does `HalvingRandomSearchCV` play in tuning an ensemble model efficiently?
* Are the search-selected hyperparameters consistent with what was actually evaluated?

---

# 2. Experimental Setup

## Dataset

* Final test set: **4,981 documents** (2,520 negative / 2,461 positive).

## Feature Extraction

Documents were transformed into numerical vectors using **CountVectorizer**.

| Configuration | Parameters                            |
| -------------- | -------------------------------------- |
| C1             | max_features=5000, ngram_range=(1,2)   |
| C2             | max_features=10000, ngram_range=(1,2)  |
| C3             | max_features=15000, ngram_range=(1,2)  |

No untuned baseline was run in this experiment — all three configurations were evaluated using tuned hyperparameters only.

---

# 3. Hyperparameter Optimization Strategy

**HalvingRandomSearchCV** was used to tune the Random Forest hyperparameters.

Unlike a plain `RandomizedSearchCV`, halving search starts by evaluating a large pool of randomly sampled candidates on a small subset of resources (e.g. a small sample of training data or few estimators), then successively eliminates the weaker-performing candidates and allocates progressively more resources (more data, more trees) to the survivors. This makes it well suited to Random Forest, where `n_estimators` itself is an expensive resource to scale — it lets many candidate configurations be screened cheaply before committing full training cost to the most promising ones.

---

# 4. Hyperparameters Explored

The search covered:

* Number of trees (`n_estimators`)
* Tree depth (`max_depth`)
* Minimum samples required for splitting (`min_samples_split`)
* Minimum samples required per leaf (`min_samples_leaf`)
* Feature selection strategy (`max_features`)
* Splitting criterion (`gini`, `entropy`, `log_loss`)
* Bootstrap sampling (`bootstrap`)

### Best Parameters Found by HalvingRandomSearchCV

| Features | n_estimators | min_samples_split | min_samples_leaf | max_features | max_depth | criterion | bootstrap |
| -------- | ------------: | ------------------: | -----------------: | ------------- | ---------- | --------- | --------- |
| 5K       | 500           | 2                   | 2                  | sqrt           | None       | entropy   | True      |
| 10K      | 300           | 5                   | 1                  | sqrt           | None       | log_loss  | False     |
| 15K      | 300           | 2                   | 1                  | sqrt           | 50         | log_loss  | False     |

### Parameters Actually Used in Final Evaluation

| Features | n_estimators | min_samples_split | min_samples_leaf | max_features | max_depth | criterion | bootstrap |
| -------- | ------------: | ------------------: | -----------------: | ------------- | ---------- | --------- | --------- |
| 5K       | 500           | 2                   | 2                  | sqrt           | None       | entropy   | True      |
| 10K      | 300           | 5                   | 1                  | sqrt           | None       | log_loss  | False     |
| 15K      | 300           | 2                   | 1                  | sqrt           | 50         | log_loss  | False     |

All three configurations were confirmed to be evaluated with exactly the parameters `HalvingRandomSearchCV` selected

---

# 5. Experimental Results

| Configuration | Accuracy   | ROC-AUC    | CV Mean | CV Std |
| -------------- | ---------- | ---------- | ------- | ------ |
| 5K Features    | 85.46%     | 0.8549     | 84.93%  | 0.00068 |
| **10K Features** | **86.13%** | **0.8614** | 85.72%  | 0.00273 |
| 15K Features   | 85.48%     | 0.8552     | 85.52%  | 0.00209 |

---

# 6. Performance Analysis

## Ensemble vs. Single Tree

Every Random Forest configuration substantially outperformed the single Decision Tree's previous best result. The weakest Random Forest run here (85.46%, 5K features) still beats the single tree's best evaluated accuracy by roughly **11 points**, confirming that aggregating many decorrelated trees meaningfully compensates for the instability and overfitting that limited the single-tree approach.

## Effect of Vocabulary Size

| Features | Accuracy   |
| -------- | ---------- |
| 5K       | 85.46%     |
| **10K**  | **86.13%** |
| 15K      | 85.48%     |

As with the Decision Tree experiments, performance peaks at 10K features and slightly declines at 15K — additional vocabulary beyond this point appears to add more noise than signal for tree-based splitting, regardless of whether a single tree or an ensemble is used.

---

# 7. Precision and Recall Analysis

### 5K Features

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.87      | 0.84   | 0.85     |
| 1     | 0.84      | 0.87   | 0.86     |

### Best Model — 10K Features

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.87      | 0.85   | 0.86     |
| 1     | 0.85      | 0.87   | 0.86     |

### 15K Features

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0     | 0.88      | 0.83   | 0.85     |
| 1     | 0.83      | 0.88   | 0.86     |

Precision and recall are far more balanced across classes here than they were for the Decision Tree — the largest recall gap for any class is around 5 points (15K, Class 0), compared to gaps of 10+ points seen in the single-tree results. This balance is a direct benefit of ensembling: averaging over many trees smooths out the class-favoring quirks any individual tree might develop.

---

# 8. ROC-AUC Analysis

| Configuration  | ROC-AUC    |
| --------------- | ---------- |
| 5K Features      | 0.8549     |
| **10K Features** | **0.8614** |
| 15K Features     | 0.8552     |

ROC-AUC mirrors the accuracy trend closely, with 10K features again the strongest performer.

---

# 9. Cross-Validation Analysis

| Configuration    | CV Mean Accuracy | CV Std Dev |
| ------------------ | ----------------- | ---------- |
| 5K Features         | 84.93%             | **0.00068** |
| **10K Features**    | **85.72%**         | 0.00273    |
| 15K Features        | 85.52%             | 0.00209    |

All three configurations show remarkably tight cross-validation spreads (all under 0.003), a sharp contrast to the Decision Tree's CV std values, which ran an order of magnitude higher in places. This is expected — ensembling naturally averages out fold-to-fold variance that a single tree is prone to. The 5K configuration is the most stable of the three, even though it isn't the most accurate.

---

# 10. Best Model Configuration

The highest-performing evaluated Random Forest was:

```python
CountVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features='sqrt',
    criterion='log_loss',
    bootstrap=False,
    random_state=0
)
```

Performance:

* Accuracy = **86.13%**
* Precision (Class 0 / 1) = **0.87 / 0.85**
* Recall (Class 0 / 1) = **0.85 / 0.87**
* ROC-AUC = **0.8614**
* Cross-Validation Accuracy = **85.72%**
* Cross-Validation Standard Deviation = **0.0027**

---

# 11. Comparison with Previously Evaluated Models (CountVectorizer)

| Model                            | Best Accuracy  | ROC-AUC |
| -------------------------------- | -------------- | ------ |
| Logistic Regression              | *88.78%*       | *0.8880* |
| LinearSVC                        | *87.51%*       | *0.8753* |
| Naive Bayes (Multinomial)        | *87.19%*       | *0.8719* |
| **Random Forest**                | **86.13%**     | **0.8614** |
| K-Nearest Neighbors              | *76.79*        | *0.7671* |
| Decision Tree                    | *74.34%*         | *0.7441*  |

*(Only Decision Tree has a confirmed CountVectorizer figure from the earlier report — send the others over and I'll complete this table.)*

---

# 12. Discussion

The core finding of this experiment is that ensembling delivers a large, consistent improvement over the single Decision Tree — roughly 11-12 accuracy points and a comparable jump in ROC-AUC — while also producing far more balanced precision/recall across classes and dramatically tighter cross-validation variance. This is consistent with the general expectation that Random Forest addresses a single tree's core weakness (high variance, easily-overfit axis-aligned splits) by averaging over many decorrelated trees.

The vocabulary-size pattern also held from the Decision Tree experiments: performance peaks at 10K features and slightly regresses at 15K, suggesting this dataset's useful signal saturates around that vocabulary size regardless of whether a single tree or an ensemble is used.

All three configurations were verified to be evaluated with the exact parameters `HalvingRandomSearchCV` selected, so the 86.13% result for 10K features can be treated as final rather than provisional.

---

# 13. Final Conclusion

This experiment evaluated Random Forest performance on binary text classification using CountVectorizer features across three vocabulary sizes, with `HalvingRandomSearchCV` used for hyperparameter tuning.

The best-evaluated model used **CountVectorizer with 10,000 features and bigrams**, paired with a **RandomForestClassifier** (300 trees, unbounded depth, sqrt features, log_loss criterion, no bootstrap), achieving **86.13% accuracy** and **0.8614 ROC-AUC** — a substantial improvement of roughly 11-12 points over the single Decision Tree's best result (74.34%).

all three configurations are now confirmed to reflect exactly what `HalvingRandomSearchCV` selected, making 86.13% a reliable Random Forest benchmark to carry forward into the combined comparison report.